# 00 - Setup Unity Catalog, Schema & Volumes

Creates the Unity Catalog catalog, schema, and volumes required by the
Nominatim import pipeline:

- **Catalog** - created if it does not exist
- **Schema** - created if it does not exist
- **`init` volume** - stores cluster init scripts (e.g. `nominatim_init.sh`)
- **`osm_data` volume** - stores downloaded OSM `.pbf` files

## Configuration

In [ ]:
dbutils.widgets.text("catalog", "justinm", "Unity Catalog name")
dbutils.widgets.text("schema", "nominatim", "Schema name")

In [ ]:
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")

print(f"Catalog: {catalog}")
print(f"Schema:  {schema}")

## Create Catalog

In [ ]:
# spark.sql(f"CREATE CATALOG IF NOT EXISTS `{catalog}`")
# print(f"Catalog '{catalog}' is ready")

## Create Schema

In [ ]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{catalog}`.`{schema}`")
print(f"Schema '{catalog}.{schema}' is ready")

## Create Volumes

In [ ]:
for volume_name in ["init", "osm_data"]:
    spark.sql(
        f"CREATE VOLUME IF NOT EXISTS `{catalog}`.`{schema}`.`{volume_name}`"
    )
    print(f"Volume '{catalog}.{schema}.{volume_name}' is ready")

## Copy Init Script to Volume

In [ ]:
import os
import shutil
from pathlib import Path

# The init script is deployed alongside this notebook in the workspace
notebook_dir = os.path.dirname(dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get())
workspace_src = f"/Workspace{notebook_dir}/nominatim_init.sh"
volume_dst = f"/Volumes/{catalog}/{schema}/init/nominatim_init.sh"

print(f"Copying init script...")
print(f"  From: {workspace_src}")
print(f"  To:   {volume_dst}")

shutil.copy2(workspace_src, volume_dst)
print("Init script copied successfully")

## Summary

In [ ]:
print("=" * 60)
print("Catalog & Volume Setup Complete")
print("=" * 60)
print()
print(f"  Catalog:    {catalog}")
print(f"  Schema:     {catalog}.{schema}")
print(f"  Init vol:   /Volumes/{catalog}/{schema}/init")
print(f"  Data vol:   /Volumes/{catalog}/{schema}/osm_data")
print(f"  Init script: /Volumes/{catalog}/{schema}/init/nominatim_init.sh")
print()
print("Next: Run 01_setup_postgis to configure PostGIS extensions")